<a href="https://colab.research.google.com/github/keshav123333/amazon-ml/blob/main/amazon_mlsecond%20notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np

In [2]:
url="http://raw.githubusercontent.com/dixitkeshav/AMAZON_ML/refs/heads/main/new/student_resource/dataset/train.csv"
df_train=pd.read_csv(url,nrows=2000)

In [3]:
df_train

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49
...,...,...,...,...
1995,61314,Item Name: Purple Yogurt Covered Raisins by It...,https://m.media-amazon.com/images/I/51S0GMYyML...,22.99
1996,207122,"Item Name: TABASCO Brand Buffalo Style Sauce, ...",https://m.media-amazon.com/images/I/71WBDCFPZg...,4.29
1997,137612,Item Name: Betty Crocker Hamburger Helper Chil...,https://m.media-amazon.com/images/I/91LSKkfmRE...,9.90
1998,106020,Item Name: RITZ Cheese Crispers Four Cheese an...,https://m.media-amazon.com/images/I/81EWQKokEx...,4.96


In [4]:
df_train['value'] = df_train['catalog_content'].str.extract(r'Value:\s*([\d.]+)')

In [5]:
df_train['value']

,value
0,72.0
1,32.0
2,11.4
3,11.25
4,12.0
...,...
1995,1.0
1996,8.6
1997,5.2
1998,42.0


In [8]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf

# -----------------------------
# 1️⃣ Image Input
# -----------------------------
img_input = Input(shape=(224, 224, 3))

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=img_input)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Fine-tune last few layers
for layer in base_model.layers[-4:]:
    layer.trainable = True

x_img = base_model.output
x_img = GlobalAveragePooling2D()(x_img)
x_img = Dense(512, activation='relu')(x_img)
x_img = BatchNormalization()(x_img)
x_img = Dropout(0.5)(x_img)

# -----------------------------
# 2️⃣ Value Input (from df["value"])
# -----------------------------
val_input = Input(shape=(1,))  # assuming single numeric column
# if you have multiple numeric columns, use shape=(num_features,)

# -----------------------------
# 3️⃣ Combine both
# -----------------------------
x = Concatenate()([x_img, val_input])

x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

output = Dense(1, activation='linear')(x)

# -----------------------------
# 4️⃣ Build Model
# -----------------------------
model = Model(inputs=[img_input, val_input], outputs=output)

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 5,105,316 (19.48 MB)

 Trainable params: 1,465,985 (5.59 MB)

 Non-trainable params: 3,639,331 (13.88 MB)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
from tqdm import tqdm
import requests
from io import BytesIO
from PIL import Image

def load_and_preprocess(img_url):
    try:
        response = requests.get(img_url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img = img.resize((224, 224))
        img_array = np.array(img)
        return preprocess_input(img_array)
    except:
        return None

# Convert all images to numpy arrays
images = []
values = []

for i, row in tqdm(df_train.iterrows(), total=len(df_train)):
    img_arr = load_and_preprocess(row['image_link'])
    if img_arr is not None:
        images.append(img_arr)
        values.append(row['value'])

X_img = np.array(images)
X_val = np.array(values).reshape(-1, 1)
y=df_train['price'].values
  # ensure shape (n,1)

# Split into train/test if needed
# from sklearn.model_selection import train_test_split
# X_img_train, X_img_test, X_val_train, X_val_test, y_train, y_test = train_test_split(X_img, X_val, y, test_size=0.2)

# Train the model
model.fit([X_img, X_val], y, epochs=10, batch_size=16)


 98%|█████████▊| 1955/2000 [05:09<00:06,  6.97it/s]

# if efficent net and text ek sath use toh

Ahh ab samajh gaya Keshav 😎 — tu basically **multi-modal regression** karna chahta hai:

* Input 1 → **Image** (EfficientNet fine-tune ho raha ho)
* Input 2 → **Text embeddings** (precomputed, e.g., MPNet 768-dim)
* Output → **Price prediction**

Yes, ye bilkul **possible hai**. Bas hume **Keras Functional API** ka use karna padega, kyunki Sequential API me multiple inputs directly handle nahi hote.

Main tujhe **step-by-step structure** bata deta hoon:

---

## 1️⃣ Image Input → EfficientNet

```python
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf

# Image input
img_input = Input(shape=(224,224,3))

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=img_input)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Fine-tune last 4 layers
for layer in base_model.layers[-4:]:
    layer.trainable = True

x_img = base_model.output
x_img = GlobalAveragePooling2D()(x_img)  # image embedding vector
x_img = Dense(512, activation='relu')(x_img)
x_img = BatchNormalization()(x_img)
x_img = Dropout(0.5)(x_img)
```

---

## 2️⃣ Text Input → Precomputed embeddings

```python
# Suppose text embeddings = 768-dim
text_input = Input(shape=(768,))
x_text = Dense(512, activation='relu')(text_input)
x_text = BatchNormalization()(x_text)
x_text = Dropout(0.5)(x_text)
```

---

## 3️⃣ Combine Image + Text

```python
combined = Concatenate()([x_img, x_text])
x = Dense(512, activation='relu')(combined)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='linear')(x)
```

---

## 4️⃣ Create Model & Compile

```python
model = Model(inputs=[img_input, text_input], outputs=output)

def smape(y_true, y_pred):
    epsilon = 1e-6
    numerator = tf.abs(y_true - y_pred)
    denominator = (tf.abs(y_true) + tf.abs(y_pred) + epsilon) / 2.0
    return 100 * tf.reduce_mean(numerator / denominator)

model.compile(optimizer='adam', loss=smape, metrics=['mae','mse'])
model.summary()
```

---

## 5️⃣ Training

```python
# Suppose X_img = images as numpy array (224,224,3)
# X_text = precomputed embeddings (768-dim)
# y = price

model.fit([X_img, X_text], y, batch_size=32, epochs=10)
```

---

### 🔹 Notes:

1. EfficientNet **last 4 layers trainable** → model images ke features update karega
2. Text embeddings **freeze** → fast aur memory efficient, agar chahe to text branch bhi trainable kar sakta hai
3. Concatenate → dono modalities ke features ek saath regression head me jaate hain
4. Agar GPU memory kam ho → fine-tune **last 2 layers** instead of 4

---

Agar tu chaahe to mai **tere existing image folder + text embeddings se full working multi-modal model ka ready-to-run code** bana du jisme EfficientNet fine-tune + text embeddings input ho, aur tu seedha train kar sake.

Chaahe mai wo bana du?
